# 03 — Capture inventory and development split manifest
Run all cells on a CPU runtime. No GPU or new dataset downloads are needed. This notebook reads notebook 02's saved file inventory from Drive and assigns whole capture files to development partitions.

**These are provisional file-disjoint splits, not validated independent-session splits.** The current dataset does not establish whether separately named captures share devices or continuous sessions. Related captures must be grouped together before final training. The three selected applications are a pilot, not a benchmark for eMBB/URLLC/mMTC.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, datetime, hashlib
import pandas as pd
PROJECT=Path('/content/drive/MyDrive/5G_QoS_Research')
# Set an explicit prior run folder here if you do not want the most recent successful run.
SOURCE_RUN=None
runs=sorted((PROJECT/'data/real_inspection').glob('*/inspection_summary.json'))
if SOURCE_RUN is None:
    candidates=[p.parent for p in runs if json.loads(p.read_text()).get('successful_previews',0)>=3 and (p.parent/'file_inventory.json').exists()]
    if not candidates: raise RuntimeError('Run notebook 02 successfully first.')
    SOURCE_RUN=candidates[-1]
SOURCE_RUN=Path(SOURCE_RUN)
files=json.loads((SOURCE_RUN/'file_inventory.json').read_text())
metadata=json.loads((SOURCE_RUN/'provider_metadata.json').read_text())
if metadata.get('currentVersionNumber',metadata.get('currentVersionNumberNullable')) != 1:
    raise RuntimeError('Dataset version changed; review before generating splits.')
print('Using saved inventory:', SOURCE_RUN)


In [ ]:
"""Deterministic development splits by capture file; not proof of independent sessions."""
from pathlib import PurePosixPath
import hashlib, json, math
from collections import defaultdict

APPLICATIONS = ('Zepeto', 'Teamfight_Tactics', 'YouTube_Live')

def build_plan(files, seed=42, applications=APPLICATIONS):
    grouped, seen = defaultdict(list), set()
    for f in files:
        name = f['name']
        if name in seen:
            raise ValueError('Duplicate inventory filename: '+name)
        seen.add(name)
        parts = PurePosixPath(name).parts
        if len(parts)<3 or not name.lower().endswith('.csv'): continue
        grouped[parts[-2]].append({'source_file':name, 'source_bytes':int(f['bytes']),
                                  'application':parts[-2], 'category':parts[-3]})
    overview = [{'application':a, 'capture_files':len(fs),
                 'source_bytes':sum(f['source_bytes'] for f in fs),
                 'eligible_for_three_way_development_split':len(fs)>=5}
                for a,fs in sorted(grouped.items())]
    rows=[]
    for app in applications:
        fs=grouped.get(app, [])
        if len(fs)<5: raise ValueError(f'{app}: need at least five files; found {len(fs)}')
        # Hash ranking is stable across inventory order and Python versions.
        ranked=sorted(fs, key=lambda f:hashlib.sha256(f"{seed}:{f['source_file']}".encode()).hexdigest())
        n_test=max(1, math.floor(len(fs)*.2))
        n_val=max(1, math.floor(len(fs)*.2))
        for i,f in enumerate(ranked):
            split='test' if i<n_test else ('validation' if i<n_test+n_val else 'train')
            rows.append(dict(f, split=split, capture_id=hashlib.sha256(f['source_file'].encode()).hexdigest()[:16],
                             session_independence='unverified', label_provenance='application directory',
                             acquisition_status='not_verified', timestamp_timezone='unknown'))
    counts={a:{s:sum(r['application']==a and r['split']==s for r in rows)
               for s in ['train','validation','test']} for a in applications}
    digest=hashlib.sha256(json.dumps(rows,sort_keys=True).encode()).hexdigest()
    summary={'stage':'capture_file_development_manifest','seed':seed,'manifest_sha256':digest,
             'selected_applications':list(applications),'capture_counts':counts,
             'capture_files':len(rows),'declared_uncompressed_bytes':sum(r['source_bytes'] for r in rows),
             'split_strategy':'within-application deterministic hash ranking of entire filenames; approximately 60/20/20',
             'file_overlap_between_partitions':False,'training_ready':False,
             'publication_ready':False,
             'limitations':['File disjointness does not establish independent sessions or devices.',
                            'No chronological or unseen-device generalization claim from this split.',
                            'Selected applications are a development subset; one application per category confounds category and application.',
                            'License and packet-level attribution remain unresolved.',
                            'Do not use the previous one-capture samples as independent train/test data.'],
             'required_before_training':['Acquire selected captures with hashes and version metadata.',
                                        'Identify recording dates, device/session provenance and possible file continuations.',
                                        'Group related captures into the SAME split and regenerate manifest if necessary.',
                                        'Detect exact/overlapping packet segments across files; review labels and flow extraction.',
                                        'Fit encoders and scalers on training only; freeze final split before tuning.']}
    return overview, rows, summary


In [ ]:
overview, manifest, summary=build_plan(files)
summary['dataset']='kimdaegyeom/5g-traffic-datasets'
summary['dataset_version']=1
summary['license']=metadata.get('licenseName',metadata.get('licenseNameNullable'))
summary['inventory_sha256']=hashlib.sha256((SOURCE_RUN/'file_inventory.json').read_bytes()).hexdigest()
from IPython.display import display
display(pd.DataFrame(overview))
display(pd.DataFrame(manifest).groupby(['application','split']).agg(captures=('capture_id','count'),source_bytes=('source_bytes','sum')))


In [ ]:
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=PROJECT/'data/manifests'/('capture_plan_'+stamp)
OUT.mkdir(parents=True,exist_ok=False)
pd.DataFrame(overview).to_csv(OUT/'application_inventory.csv',index=False)
pd.DataFrame(manifest).to_csv(OUT/'capture_split_manifest.csv',index=False)
(OUT/'capture_split_manifest.json').write_text(json.dumps(manifest,indent=2))
# This review template contains no invented session/device metadata.
review=pd.DataFrame(manifest)[['capture_id','source_file','application','split']].copy()
for column in ['recording_start','recording_end','timezone','device_group','session_group','continuation_of','label_evidence','review_notes']:
    review[column]=''
review.to_csv(OUT/'capture_provenance_review.csv',index=False)
(OUT/'split_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Saved:', OUT)


## Send back
Attach **split_summary.json** from the printed Drive folder. Do not manually guess the empty provenance fields. Those fields require capture inspection or provider documentation.

This notebook intentionally performs no random packet-row split, no window generation, no downloads and no training. Next comes capture acquisition and provenance/overlap validation. That stage may revise these provisional assignments. Final test results must not drive split selection.
